# Avaliação dos modelos no conjunto de teste

Este notebook carrega os modelos já treinados em `artifacts/models`, reconstrói o mesmo conjunto de teste temporal usado no treinamento (junho a dezembro de 2019) e gera uma avaliação comparável entre todos os modelos.

As saídas são salvas automaticamente em:

- `reports/metrics/modeling`: métricas e dados dos gráficos em CSV;
- `reports/figures/modeling/evaluation`: gráficos em PNG (200 dpi).

A classe positiva é **1 = dengue confirmada**. Precisão, recall e F1 referem-se a essa classe.

## 1. Configuração

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dengue_pipeline").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline import DengueDataCleaner
from dengue_pipeline.paths import MODELS_DIR, MODEL_FIGURES_DIR

METRICS_DIR = PROJECT_ROOT / "reports" / "metrics" / "modeling"
EVALUATION_FIGURES_DIR = MODEL_FIGURES_DIR / "evaluation"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    "MLP": "mlp.joblib",
    "XGBoost": "xgboost.joblib",
    "LightGBM": "lightgbm.joblib",
}

MODEL_COLORS = {
    "MLP": "#2563eb",
    "XGBoost": "#0f766e",
    "LightGBM": "#7c3aed",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 10,
    "font.size": 10,
})

print(f"Modelos: {MODELS_DIR}")
print(f"Tabelas: {METRICS_DIR}")
print(f"Figuras: {EVALUATION_FIGURES_DIR}")

## 2. Reconstrução do conjunto de teste

A preparação abaixo repete o fluxo do notebook `models.ipynb`: transformação dos dados, recorte temporal e remoção das mesmas colunas administrativas. Nenhum registro de treino é usado no cálculo das métricas.

In [ ]:
TARGET = "final_classification"
ADMINISTRATIVE_FEATURES = [
    "notification_year",
    "notification_month",
    "symptom_epi_year",
    "notif_municipality",
    "notif_health_region",
    "health_facility",
]

cleaner = DengueDataCleaner()
df = cleaner.transformar_ml()

test_mask = (df["notification_year"] == 2019) & (df["notification_month"] >= 6)
df_test = df[test_mask].copy()

y_test = df_test[TARGET].astype(int)
X_test = df_test.drop(columns=[TARGET])
X_test = X_test.select_dtypes(include=["number"])
X_test = X_test.drop(columns=ADMINISTRATIVE_FEATURES, errors="ignore")
X_test = X_test.astype("float32")

test_summary = pd.DataFrame({
    "item": [
        "data_source",
        "period_start",
        "period_end",
        "n_samples",
        "n_features",
        "n_confirmed",
        "n_discarded",
        "positive_rate",
    ],
    "value": [
        "data/raw/parquet/DENGBR*.parquet",
        "2019-06",
        "2019-12",
        len(y_test),
        X_test.shape[1],
        int(y_test.sum()),
        int((y_test == 0).sum()),
        float(y_test.mean()),
    ],
})
test_summary.to_csv(METRICS_DIR / "test_set_summary.csv", index=False, encoding="utf-8-sig")

print(f"Teste: {X_test.shape[0]:,} registros e {X_test.shape[1]} features")
print(f"Confirmados: {y_test.sum():,} ({y_test.mean():.2%})")

# A base completa é grande, então ela não precisa continuar na memória durante as previsões.
del cleaner, df, df_test

test_summary

## 3. Carregamento e avaliação

Métricas calculadas com o limiar base de **0,07**:

- **accuracy**: proporção total de acertos;
- **balanced_accuracy**: média do recall das duas classes;
- **precision**: entre os casos previstos como dengue, quantos foram confirmados;
- **recall / sensitivity**: entre os casos confirmados, quantos foram detectados;
- **specificity**: entre os descartados, quantos foram corretamente descartados;
- **f1**: média harmônica de precisão e recall;
- **roc_auc** e **pr_auc**: discriminação usando todos os limiares.

In [ ]:
def pegar_probabilidade_positiva(modelo, dados):
    probabilidades = np.array(modelo.predict_proba(dados))
    if probabilidades.ndim == 2:
        probabilidades = probabilidades[:, 1]
    return probabilidades


def escolher_pontos(total, limite=1000):
    if total > limite:
        return np.linspace(0, total - 1, limite).astype(int)
    return np.arange(total)


BASE_THRESHOLD = 0.07

metric_rows = []
report_frames = []
confusion_rows = []
roc_frames = []
pr_frames = []
threshold_frames = []
evaluation_results = {}

thresholds_to_test = [0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

for model_name, filename in MODEL_FILES.items():
    model_path = MODELS_DIR / filename
    print(f"Avaliando {model_name}...")
    model = joblib.load(model_path)

    if hasattr(model, "feature_names"):
        features = model.feature_names
    else:
        features = model.feature_names_in_
    X_model = X_test[features]

    y_score = pegar_probabilidade_positiva(model, X_model)
    y_pred = (y_score >= BASE_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    metric_rows.append({
        "model": model_name,
        "threshold": BASE_THRESHOLD,
        "n_test": len(y_test),
        "positive_rate": y_test.mean(),
        "predicted_positive_rate": y_pred.mean(),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": average_precision_score(y_test, y_score),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })

    report_dict = classification_report(y_test, y_pred, target_names=["Descartado (0)", "Confirmado (1)"], output_dict=True, zero_division=0)
    report = pd.DataFrame(report_dict).T.reset_index().rename(columns={"index": "class"})
    report.insert(0, "model", model_name)
    report_frames.append(report)

    for actual, predicted, count in [(0, 0, tn), (0, 1, fp), (1, 0, fn), (1, 1, tp)]:
        confusion_rows.append({
            "model": model_name,
            "actual": actual,
            "predicted": predicted,
            "count": int(count),
        })

    fpr, tpr, roc_thresholds = roc_curve(y_test, y_score)
    roc_points = escolher_pontos(len(fpr))
    roc_frames.append(pd.DataFrame({
        "model": model_name,
        "false_positive_rate": fpr[roc_points],
        "true_positive_rate": tpr[roc_points],
        "threshold": roc_thresholds[roc_points],
    }))

    pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_test, y_score)
    pr_thresholds_complete = np.append(pr_thresholds, np.nan)
    pr_points = escolher_pontos(len(pr_recall))
    pr_frames.append(pd.DataFrame({
        "model": model_name,
        "recall": pr_recall[pr_points],
        "precision": pr_precision[pr_points],
        "threshold": pr_thresholds_complete[pr_points],
    }))

    threshold_rows = []
    for threshold in thresholds_to_test:
        threshold_pred = (y_score >= threshold).astype(int)
        threshold_rows.append({
            "model": model_name,
            "threshold": threshold,
            "accuracy": accuracy_score(y_test, threshold_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, threshold_pred),
            "precision": precision_score(y_test, threshold_pred, zero_division=0),
            "recall": recall_score(y_test, threshold_pred, zero_division=0),
            "f1": f1_score(y_test, threshold_pred, zero_division=0),
            "predicted_positive_rate": threshold_pred.mean(),
        })
    threshold_frames.append(pd.DataFrame(threshold_rows))

    evaluation_results[model_name] = {"confusion_matrix": np.array([[tn, fp], [fn, tp]])}

metrics_df = pd.DataFrame(metric_rows).sort_values("roc_auc", ascending=False)
classification_report_df = pd.concat(report_frames, ignore_index=True)
confusion_matrices_df = pd.DataFrame(confusion_rows)
roc_curves_df = pd.concat(roc_frames, ignore_index=True)
precision_recall_curves_df = pd.concat(pr_frames, ignore_index=True)
threshold_metrics_df = pd.concat(threshold_frames, ignore_index=True)

tables = {
    "model_metrics.csv": metrics_df,
    "classification_report.csv": classification_report_df,
    "confusion_matrices.csv": confusion_matrices_df,
    "roc_curve_points.csv": roc_curves_df,
    "precision_recall_curve_points.csv": precision_recall_curves_df,
    "threshold_metrics.csv": threshold_metrics_df,
}
for filename, table in tables.items():
    table.to_csv(METRICS_DIR / filename, index=False, encoding="utf-8-sig")

display_columns = [
    "model", "accuracy", "balanced_accuracy", "precision", "recall",
    "specificity", "f1", "roc_auc", "pr_auc",
]
metrics_df[display_columns].round(4)

## 4. Comparação das principais métricas

In [ ]:
metric_labels = {
    "accuracy": "Acurácia",
    "balanced_accuracy": "Acurácia\nbalanceada",
    "precision": "Precisão",
    "recall": "Recall",
    "f1": "F1-score",
    "roc_auc": "ROC AUC",
    "pr_auc": "PR AUC",
}

plot_data = metrics_df.set_index("model")[list(metric_labels)].rename(
    columns=metric_labels
)

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(plot_data.columns))
bar_width = 0.24
center = (len(MODEL_FILES) - 1) / 2

for index, model_name in enumerate(MODEL_FILES):
    values = plot_data.loc[model_name].values
    bars = ax.bar(
        x + (index - center) * bar_width,
        values,
        width=bar_width,
        label=model_name,
        color=MODEL_COLORS[model_name],
        alpha=0.9,
    )
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.012,
            f"{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#334155",
            rotation=90,
        )

ax.set_title("Comparação das Principais Métricas no Conjunto de Teste", pad=16)
ax.set_xticks(x)
ax.set_xticklabels(plot_data.columns)
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylabel("Resultado")
ax.grid(axis="y", linestyle="--", alpha=0.22)
ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=2, loc="upper center")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="both", length=0)

fig.tight_layout()
fig.savefig(
    EVALUATION_FIGURES_DIR / "model_metrics_comparison.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 5. Curvas ROC e Precision–Recall

A curva ROC compara sensibilidade e taxa de falsos positivos. A curva Precision–Recall dá mais visibilidade ao desempenho sobre a classe positiva e costuma ser mais informativa quando há desbalanceamento.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for model_name in MODEL_FILES:
    curve = roc_curves_df[roc_curves_df["model"] == model_name]
    auc_value = metrics_df.set_index("model").loc[model_name, "roc_auc"]
    ax.plot(
        curve["false_positive_rate"],
        curve["true_positive_rate"],
        color=MODEL_COLORS[model_name],
        linewidth=2.2,
        label=f"{model_name} (AUC = {auc_value:.3f})",
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="#94a3b8", linewidth=1.3, label="Aleatório")
ax.set_title("Curvas ROC dos Modelos", pad=14)
ax.set_xlabel("Taxa de falsos positivos (1 - especificidade)")
ax.set_ylabel("Taxa de verdadeiros positivos (recall)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(linestyle="--", alpha=0.22)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower right")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="both", length=0)
fig.tight_layout()
fig.savefig(EVALUATION_FIGURES_DIR / "roc_curves.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


fig, ax = plt.subplots(figsize=(9, 7))
for model_name in MODEL_FILES:
    curve = precision_recall_curves_df[
        precision_recall_curves_df["model"] == model_name
    ]
    auc_value = metrics_df.set_index("model").loc[model_name, "pr_auc"]
    ax.plot(
        curve["recall"],
        curve["precision"],
        color=MODEL_COLORS[model_name],
        linewidth=2.2,
        label=f"{model_name} (AP = {auc_value:.3f})",
    )

prevalence = y_test.mean()
ax.axhline(
    prevalence,
    linestyle="--",
    color="#94a3b8",
    linewidth=1.3,
    label=f"Baseline ({prevalence:.1%})",
)
ax.set_title("Curvas Precision–Recall dos Modelos", pad=14)
ax.set_xlabel("Recall")
ax.set_ylabel("Precisão")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(linestyle="--", alpha=0.22)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower left")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="both", length=0)
fig.tight_layout()
fig.savefig(
    EVALUATION_FIGURES_DIR / "precision_recall_curves.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 6. Matrizes de confusão

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_FILES), figsize=(16, 5))
axes = np.atleast_1d(axes)

for ax, model_name in zip(axes, MODEL_FILES):
    matrix = evaluation_results[model_name]["confusion_matrix"]
    row_totals = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(
        matrix,
        row_totals,
        out=np.zeros_like(matrix, dtype=float),
        where=row_totals != 0,
    )
    image = ax.imshow(normalized, cmap="Blues", vmin=0, vmax=1)

    for row in range(2):
        for column in range(2):
            text_color = "white" if normalized[row, column] > 0.52 else "#0f172a"
            ax.text(
                column,
                row,
                f"{matrix[row, column]:,}".replace(",", ".")
                + f"\n({normalized[row, column]:.1%})",
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color=text_color,
            )

    ax.set_title(model_name, fontsize=12, pad=10)
    ax.set_xticks([0, 1], labels=["Descartado", "Confirmado"])
    ax.set_yticks([0, 1], labels=["Descartado", "Confirmado"])
    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe real")
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle("Matrizes de Confusão no Conjunto de Teste", fontsize=16, fontweight="bold", y=1.01)
fig.colorbar(image, ax=axes, fraction=0.025, pad=0.03, label="Proporção dentro da classe real")
fig.savefig(
    EVALUATION_FIGURES_DIR / "confusion_matrices.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 7. Sensibilidade ao limiar de decisão

O limiar não deve ser escolhido usando o conjunto de teste final. Este gráfico serve para entender o comportamento dos modelos; uma escolha operacional de limiar deve ser feita em validação separada e considerar o custo de falsos negativos e falsos positivos.

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_FILES), figsize=(16, 5), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for index, (ax, model_name) in enumerate(zip(axes, MODEL_FILES)):
    data = threshold_metrics_df[threshold_metrics_df["model"] == model_name]
    ax.plot(data["threshold"], data["precision"], label="Precisão", color="#2563eb", linewidth=3)
    ax.plot(data["threshold"], data["recall"], label="Recall", color="#dc2626", linewidth=3)
    ax.plot(data["threshold"], data["f1"], label="F1-score", color="#0f766e", linewidth=3)
    ax.axvline(BASE_THRESHOLD, color="#94a3b8", linestyle="--", linewidth=1.2)
    ax.set_title(model_name, fontsize=12, pad=10)
    ax.set_xlim(0.05, 0.95)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Limiar")
    if index == 0:
        ax.set_ylabel("Resultado")
    ax.grid(linestyle="--", alpha=0.22)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.01))
fig.suptitle("Precisão, Recall e F1 por Limiar de Decisão", fontsize=16, fontweight="bold", y=1.06)
fig.tight_layout()
fig.savefig(
    EVALUATION_FIGURES_DIR / "threshold_analysis.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

## 8. Arquivos gerados

In [ ]:
generated_files = sorted(METRICS_DIR.glob("*.csv")) + sorted(EVALUATION_FIGURES_DIR.glob("*.png"))
pd.DataFrame({
    "arquivo": [str(path.relative_to(PROJECT_ROOT)) for path in generated_files],
    "tamanho_kb": [round(path.stat().st_size / 1024, 1) for path in generated_files],
})